# Notebook 3: Stage-2 Disease Classifiers (per crop)

Requires Notebook 2's output (`stage1_crop_classifier.pt` on Drive) plus Notebook 1's `data.zip`. Trains one disease classifier per crop (6 total), each warm-started from Stage-1's domain-adapted backbone. Loops over crops in one notebook rather than six separate ones, with per-crop resumability so a GPU-quota interruption doesn't lose progress on crops already finished.

This is the harder half of the pipeline — Stage-1 (crop type) got ~100% because crop types look very different from each other. Disease-within-a-crop is a subtler visual distinction, expect real errors here.

## Task 1: Setup — load Notebook 1's data + Notebook 2's Stage-1 checkpoint

In [ ]:
!pip install -q timm pillow pandas scikit-learn

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import zipfile
import pandas as pd

DRIVE_ZIP = "/content/drive/MyDrive/crop_disease/data.zip"
UNIFIED_ROOT = "/content/data"
STAGE1_CKPT = "/content/drive/MyDrive/crop_disease/stage1_crop_classifier.pt"

assert os.path.exists(DRIVE_ZIP), f"{DRIVE_ZIP} not found — run Notebook 1 first"
assert os.path.exists(STAGE1_CKPT), f"{STAGE1_CKPT} not found — run Notebook 2 first"

os.makedirs(UNIFIED_ROOT, exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as zf:
    zf.extractall(UNIFIED_ROOT)

manifest = pd.read_csv(os.path.join(UNIFIED_ROOT, "manifest.csv"))
print(manifest.shape)
print(manifest.groupby(["crop", "disease"]).size())

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

stage1_ckpt = torch.load(STAGE1_CKPT, map_location=DEVICE)
CROP_TO_IDX = stage1_ckpt["crop_to_idx"]
CROPS = sorted(CROP_TO_IDX, key=CROP_TO_IDX.get)
NUM_CROPS = len(CROPS)
print("Crops (Stage-1 order):", CROP_TO_IDX)

## Task 2: Shared dataset / transform / train utilities

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class DiseaseDataset(Dataset):
    def __init__(self, df, label_to_idx, transform):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        label = self.label_to_idx[row["disease"]]
        return self.transform(img), label

In [ ]:
import re
import timm
import torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit

STAGE2_EPOCHS = 8
STAGE2_DIR = "/content/drive/MyDrive/crop_disease"

# Several source datasets are pre-augmented (rotated/zoomed/cropped copies of
# the same base leaf photo, e.g. "resized_yellow (179).jpeg" / "rotated_yellow
# (179).jpeg"). A plain random split can put augmented siblings of the same
# photo in both train and val, inflating val accuracy. Strip the known prefixes
# to recover a shared "source photo" key, then split by that key (grouped) so
# siblings always land on the same side of the split. Namespaced by disease
# because the citrus source reuses identical filenames (e.g. "Image (36).png")
# across different disease folders — without the namespace those unrelated
# images would get wrongly merged into one group.
AUG_PREFIX_RE = re.compile(
    r"^(?:[a-z0-9]+_primary_|multi_crop_supplement_)"
    r"(?:resized_|rotated_|zoomed_|cropped_|flipped_horiz_|flipped_vert_|flipped_)*",
    re.IGNORECASE,
)

def build_group_key(filepath, disease):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    stripped = AUG_PREFIX_RE.sub("", stem, count=1)
    return f"{disease}::{stripped.lower()}"

def train_disease_head(crop_name, crop_df):
    diseases = sorted(crop_df["disease"].unique())
    disease_to_idx = {d: i for i, d in enumerate(diseases)}
    ckpt_path = os.path.join(STAGE2_DIR, f"stage2_{crop_name}_disease.pt")

    crop_df = crop_df.copy()
    crop_df["group_key"] = [build_group_key(fp, d) for fp, d in zip(crop_df["filepath"], crop_df["disease"])]
    n_groups, n_images = crop_df["group_key"].nunique(), len(crop_df)
    dup_note = f"({n_images - n_groups} likely augmentation duplicates found)" if n_groups < n_images else "(no duplicates detected by filename)"
    print(f"  [{crop_name}] {n_images} images -> {n_groups} distinct source-photo groups {dup_note}")

    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    train_idx, val_idx = next(gss.split(crop_df, groups=crop_df["group_key"]))
    train_df, val_df = crop_df.iloc[train_idx], crop_df.iloc[val_idx]

    train_ds = DiseaseDataset(train_df, disease_to_idx, train_tfms)
    val_ds = DiseaseDataset(val_df, disease_to_idx, eval_tfms)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, drop_last=len(train_ds) > 32)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

    class_counts = train_df["disease"].map(disease_to_idx).value_counts().sort_index()
    weights = (1.0 / class_counts.reindex(range(len(diseases)), fill_value=1)).values
    class_weights = torch.tensor(weights * len(diseases) / weights.sum(), dtype=torch.float32).to(DEVICE)

    model = timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=NUM_CROPS)
    model.load_state_dict(stage1_ckpt["model_state"])
    model.reset_classifier(num_classes=len(diseases))
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, STAGE2_EPOCHS * len(train_loader)))
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    start_epoch = 0
    best_val_acc = 0.0
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        if ckpt.get("diseases") == diseases and ckpt.get("split_version") == "group_v1":
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            start_epoch = ckpt["epoch"] + 1
            best_val_acc = ckpt.get("val_acc", 0.0)
            print(f"  [{crop_name}] resumed from epoch {start_epoch}, best_val_acc={best_val_acc:.4f}")
        else:
            print(f"  [{crop_name}] existing checkpoint is from the old (leaky) split — retraining from scratch")

    if start_epoch >= STAGE2_EPOCHS:
        print(f"  [{crop_name}] already fully trained ({STAGE2_EPOCHS} epochs) on the group-aware split, skipping")
        return

    for epoch in range(start_epoch, STAGE2_EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += imgs.size(0)
        val_acc = val_correct / val_total

        print(f"  [{crop_name}] epoch {epoch+1}/{STAGE2_EPOCHS} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        best_val_acc = max(best_val_acc, val_acc)
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": val_acc,
            "diseases": diseases,
            "split_version": "group_v1",
        }, ckpt_path)

    print(f"  [{crop_name}] done. best_val_acc={best_val_acc:.4f}. Checkpoint: {ckpt_path}")

## Task 3: Train all 6 per-crop disease heads

In [ ]:
for crop_name in CROPS:
    print(f"\n=== {crop_name} ===")
    crop_df = manifest[manifest["crop"] == crop_name]
    train_disease_head(crop_name, crop_df)

If this cell stops partway (GPU quota, disconnect), just rerun it — `train_disease_head` skips crops whose checkpoint is already at `STAGE2_EPOCHS` and resumes any crop that's partway through.

## Task 4: Evaluation report per crop

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit

for crop_name in CROPS:
    ckpt_path = os.path.join(STAGE2_DIR, f"stage2_{crop_name}_disease.pt")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    diseases = ckpt["diseases"]
    disease_to_idx = {d: i for i, d in enumerate(diseases)}

    crop_df = manifest[manifest["crop"] == crop_name].copy()
    crop_df["group_key"] = [build_group_key(fp, d) for fp, d in zip(crop_df["filepath"], crop_df["disease"])]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    _, val_idx = next(gss.split(crop_df, groups=crop_df["group_key"]))
    val_df = crop_df.iloc[val_idx]
    val_loader = DataLoader(DiseaseDataset(val_df, disease_to_idx, eval_tfms), batch_size=32, shuffle=False, num_workers=2)

    model = timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=len(diseases))
    model.load_state_dict(ckpt["model_state"])
    model = model.to(DEVICE).eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print(f"\n=== {crop_name} (val_acc during training: {ckpt['val_acc']:.4f}) ===")
    print(classification_report(all_labels, all_preds, target_names=diseases, zero_division=0))

## Task 5: Verify all 6 checkpoints load standalone

In [ ]:
for crop_name in CROPS:
    ckpt_path = os.path.join(STAGE2_DIR, f"stage2_{crop_name}_disease.pt")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    diseases = ckpt["diseases"]
    model = timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=len(diseases))
    model.load_state_dict(ckpt["model_state"])
    model = model.to(DEVICE).eval()
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        out = model(dummy)
    assert out.shape == (2, len(diseases)), f"{crop_name}: expected (2, {len(diseases)}), got {out.shape}"
    print(f"{crop_name}: OK, {len(diseases)} classes {diseases}")